# Permutation Assembly Flow-Shop Scheduling Problem - Benchmarks

This notebook solves benchmark instances of the permutation assembly
flow-shop scheduling problem (PAFSSP) with different machine layouts.
See the [PAFSSP tutorial](pafssp.ipynb) for the problem formulation.

Instances are available in the `data/pafssp` folder (see its README),
as JSON files with processing times `p[j][m]` and machine precedence
`edges`.

In [ ]:
import json
import random
import time

from bnbprob.pafssp import (
    CallbackBnB,
    CycleBestFlowShop,
    PermFlowShop,
    plot_gantt,
)


def load(path: str) -> tuple[list[list[int]], list[tuple[int, int]]]:
    with open(path, 'r', encoding='utf8') as f:
        data = json.load(f)
    return data['p'], [tuple(e) for e in data['edges']]


def solve(bnb: CallbackBnB, **kwargs):
    start = time.time()
    sol = bnb.solve(**kwargs)
    print(f"Solved in {time.time() - start:.1f} s ({bnb.explored} nodes explored)")
    print(sol)
    return sol

## Assembly-Disassembly Chain

A chain layout of 17 machines with assembly and disassembly operations,
and 20 jobs.

The node manager `CycleBestFlowShop` is a cyclic best-first search over
the levels of the tree. With a negative `max_size` it starts in its
fallback mode (deepest non-empty level first), and with
`permanent_fallback=True` it never leaves it: a depth-first search in
which nodes of the same level are dequeued by best bound (and least
idle time as a tie-breaker). Timings are illustrative and depend on the
hardware.

In [ ]:
p, edges = load('../data/pafssp/fad5-chain/fad5-043-50-chain10.json')
print(f"{len(p)} jobs; {len(p[0])} machines")
print(edges)

problem = PermFlowShop.from_p(p, edges=edges, constructive='iga')

In [ ]:
ws = problem.warmstart()
ws.calc_bound()

In [ ]:
ws.calc_bound()

In [ ]:
p, edges = load('../data/pafssp/fad5-chain/fad5-043-50-chain10.json')
print(f"{len(p)} jobs; {len(p[0])} machines")
print(edges)

problem = PermFlowShop.from_p(p, edges=edges, constructive='neh')
bnb = CallbackBnB(problem)
bnb.set_manager(CycleBestFlowShop(-1, permanent_fallback=True))
sol = solve(bnb, timelimit=3600)

In [ ]:
plot_gantt(sol.problem.sequence, dpi=120, seed=42, figsize=[8, 4])

## Multi-Assembly Chain

Several lines of machines joined by assembly operations, with 20 jobs
on 17 machines. Here the warmstart is the Iterated Greedy Algorithm
(`constructive='iga'`).

In [ ]:
p, edges = load('../data/pafssp/ma-chain/ma-001-20-chain7.json')
print(f"{len(p)} jobs; {len(p[0])} machines")
print(edges)

problem = PermFlowShop.from_p(p, edges=edges, constructive='iga')
bnb = CallbackBnB(problem)
sol = solve(bnb, timelimit=600)

In [ ]:
plot_gantt(sol.problem.sequence, dpi=120, seed=42, figsize=[8, 6])

## Custom Layout

Any directed acyclic graph of machines can be used. Here, a random
instance with 10 jobs on 8 machines, in which machines 1 and 3 have
several predecessors (assembly) and several successors (disassembly).

In [ ]:
rng = random.Random(12)

edges = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),
    (5, 1),
    (1, 6),
    (6, 3),
    (3, 7),
]
p = [[rng.randint(1, 100) for _ in range(8)] for _ in range(10)]

problem = PermFlowShop.from_p(p, edges=edges, constructive='iga')
bnb = CallbackBnB(problem)
sol = solve(bnb, timelimit=600)

In [ ]:
plot_gantt(sol.problem.sequence, dpi=120, seed=42, figsize=[8, 6])

## MILP Model

For comparison, the positional MILP model is available in
`bnbprob.pafssp.mip` (requires `pyomo` and a solver, such as `highspy`
or `gurobipy`):

```python
import pyomo.environ as pyo

from bnbprob.pafssp.mip import positional_model

model = positional_model(p, edges)

# HiGHS
solver = pyo.SolverFactory("appsi_highs")
solver.options["mip_heuristic_effort"] = 0.1
solver.options["time_limit"] = 120
solver.solve(model, tee=True)

# Gurobi
solver = pyo.SolverFactory("gurobi", solver_io="python")
solver.options["Heuristics"] = 0.2
solver.options["Cuts"] = 2
solver.options["TimeLimit"] = 120
solver.solve(model, tee=True)
```